# Offline ALNS Repair Model Training Pipeline

This notebook documents the complete training workflow for learning a repair policy in the Hybrid ALNS solver using offline supervised learning.

## Overview

The pipeline trains a GradientBoosting classifier to rank feasible bin repair actions during ALNS search. The workflow follows three main stages:
1. **Baseline training** on synthetic instances
2. **Collection** of realistic states from actual ALNS trajectories
3. **Augmented retraining** on synthetic + collected data

All model artifacts are saved as pickle files for deployment in the solver.

## 1. Problem Definition

At ALNS step $t$, a partial solution state is $x_t$. For each feasible bin assignment action $a \in \mathcal{A}(x_t)$, the model predicts a score used to rank actions. The optimizer then favors the highest-ranked repair actions.

**Key insight**: We optimize for **ranking quality** (ROC-AUC / Average Precision), not just binary classification accuracy, because ALNS uses model scores to prioritize candidate bins.

**Deliverables**:
- `repair_model_v1.pkl` – baseline model trained on synthetic data
- `alns_states_v1.pkl` – collected states from real ALNS trajectories
- `repair_model_v2.pkl` – augmented model trained on merged data

## 2. Data Augmentation Strategy

Synthetic data alone under-represents states encountered during real ALNS search. To reduce this distribution mismatch, we collect additional realistic states and retrain on the merged dataset.

**Key parameters**:
- `--instances`: total number of sampled trajectories or instances
- `--n-min`, `--n-max`: instance size range
- `--max-negatives`: negative sampling budget (class balance control)
- `--iterations`: ALNS trajectory length per collection run
- `--seed`: reproducibility

## 3. Training Pipeline

### Step 1: Baseline Model Training

Train the initial model on synthetic instances using 5-fold cross-validation.

**Expected output**: `repair_model_v1.pkl`

In [ ]:
!python train_repair_model.py \
    --instances 4000 \
    --n-min 50 --n-max 200 \
    --max-negatives 5 \
    --seed 0 \
    --workers 1 \
    --output repair_model_v1.pkl \
    --cv-folds 5

#### Execution & Results

**Dataset generation**
- 4,000 synthetic instances
- 1,313,891 rows, 11 features
- Positive rate: 0.2194

**Training**
- GradientBoosting (180 estimators, depth 4, lr 0.05, subsample 0.8)
- Duration: 1,696.0s
- 5-fold CV ROC-AUC: 0.8769 ± 0.0003

**Test metrics**
- ROC-AUC: 0.8790
- Average Precision: 0.7306
- F1: 0.6459
- Precision: 0.5530
- Recall: 0.7763

### Step 2: Collect Real ALNS States

Use the baseline model to collect realistic repair states from actual ALNS trajectories.

**Expected output**: `alns_states_v1.pkl`

In [ ]:
!python collect_alns_states.py \
    --model-path repair_model_v1.pkl \
    --instances 500 \
    --n-min 50 --n-max 200 \
    --iterations 200 \
    --max-negatives 5 \
    --seed 1 \
    --output alns_states_v1.pkl

#### Execution & Results

**Collection summary**
- 500 ALNS runs
- 309,539 collected rows
- Positive rate: 0.297

**Output**
- `alns_states_v1.pkl`

This augmented dataset is large enough to meaningfully shift training toward real search states.

### Step 3: Augmented Model Retraining

Merge collected ALNS states with synthetic data and retrain for improved ranking quality.

**Expected output**: `repair_model_v2.pkl` (final model for deployment)

In [ ]:
!python train_repair_model.py \
    --instances 2000 \
    --n-min 50 --n-max 200 \
    --max-negatives 3 \
    --seed 0 \
    --workers 2 \
    --augment-with alns_states_v1.pkl \
    --output repair_model_v2.pkl \
    --cv-folds 3 \
    --no-plots \
    --no-learning-curves

#### Execution & Results

**Merged training data**
- Augmented with 309,539 ALNS rows
- Effective dataset after processing: 492,710 rows, 11 features

**Training**
- GradientBoosting (500 estimators, depth 6, lr 0.03, subsample 0.75)
- Duration: 3,638.4s
- 3-fold CV ROC-AUC: 0.9090 ± 0.0005

**Test metrics**
- ROC-AUC: 0.9098
- Average Precision: 0.8381
- F1: 0.7444
- Precision: 0.6934
- Recall: 0.8035

**Confusion matrix**
- TP: 28,485
- FP: 12,596
- FN: 6,967
- TN: 72,290

**Summary**: Ranking quality improved significantly from v1 to v2 (0.8790 → 0.9098 ROC-AUC), confirming augmentation strategy effectiveness.

## 4. Model Performance Comparison

| Metric | v1 (Baseline) | v2 (Augmented) | Improvement |
|--------|--------------|----------------|------------|
| **ROC-AUC** | 0.8790 | 0.9098 | +3.1% |
| **Avg Precision** | 0.7306 | 0.8381 | +14.7% |
| **F1-Score** | 0.6459 | 0.7444 | +15.2% |
| **Precision** | 0.5530 | 0.6934 | +25.4% |
| **Recall** | 0.7763 | 0.8035 | +3.5% |
| **Training Samples** | 1,313,891 | 492,710* | - |

*v2 used only 2,000 synthetic instances augmented with 309,539 collected ALNS states, resulting in effective 492,710 samples after balancing.

**Conclusion**: The augmentation strategy significantly improves ranking quality (primary metric for ALNS deployment) while using fewer total training samples. Model v2 is preferred for deployment.

In [ ]:
!python ..\..\..\utilities\benchmarking.py \
    --solver hybrid_ml_metaheuristics/hybrid_alns/solver.py \
    --dataset falkenauer-u \
    --method-args "model_path=models/repair_model_v2.pkl"